In [ ]:
from torch.utils.data import Dataset 
from PIL import Image
import os 
import torch 

class SegmentationDataLoader(Dataset): 
    def __init__(
        self, 
        cell_types: list[str], 
        augmentation_factor: int = 1,  
        set_type: str = "train", 
        transform=None
    ): 
        self.cell_types = cell_types 
        self.set_type = set_type 
        self.images: list[tuple] = []
        self.transform = transform 
        self.augmentation_factor = augmentation_factor

        for cell_type in self.cell_types: 
            for movie_id in ["01", "02"]: 
            
                cell_path: str = os.path.join("..", "..", self.set_type, cell_type, movie_id) 
                segment_gt_path: str = os.path.join("..", "..", self.set_type, cell_type, f"{movie_id}_GT", "SEG")

                for image_name in os.listdir(segment_gt_path): 
                    cell_segment_image_path: str = os.path.join(segment_gt_path, image_name)
                    image_number: str = image_name.split("man_seg")[1].split(".tif")[0]
                    cell_image_path: str = os.path.join(cell_path, f"t{image_number}.tif")

                    self.images.append((
                        cell_type, 
                        cell_image_path,
                        cell_segment_image_path, 
                        movie_id 
                    ))
    
    def __len__(self): 
        return len(self.images) * self.augmentation_factor
    
    def __getitem__(self, index: int): 
        real_index = index % len(self.images)
        cell_type, cell_image_path, cell_segment_image_path, movie_id = self.images[real_index]
        cell_image = Image.open(cell_image_path).convert("L")
        cell_segment_image = Image.open(cell_segment_image_path).convert("L")

        if self.transform:
            cell_image, cell_segment_image = self.transform(cell_image, cell_segment_image)

        return (cell_type, cell_image, cell_segment_image, movie_id)
     

In [2]:
from torchvision.transforms import v2 
augmentation_factor = 3

probs = 1 - (1 - (augmentation_factor - 1) 
         / augmentation_factor) ** 0.25 

transform = v2.Compose([
    v2.RandomApply(
        transforms=[
            v2.ElasticTransform(
                alpha=90, 
                sigma=8, 
            )
        ], 
        p = probs
    ), 

    v2.RandomApply(
        transforms = [
            v2.RandomAffine(
                degrees=10, 
                translate=(0.1, 0.1), 
                scale=(0.9, 1.1), 
                shear=10
            ), 
        ], 
        p = probs, 
    ), 
    
    v2.RandomApply(
        transforms = [
            v2.ColorJitter(
                brightness=0.2, 
                contrast=0.2,
            )
        ], p = probs 
    ), 

    v2.RandomApply(
        transforms = [
            v2.RandomRotation(20)
        ], p = probs 
    ), 

    v2.Compose([
        v2.ToImage(), 
        v2.ToDtype(torch.float32, scale=True)
    ])

])

In [ ]:
from torch.utils.data import DataLoader

train_dataset = SegmentationDataLoader(
    cell_types=["DIC-C2DH-HeLa"],
    set_type="train",
    transform=transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True,
)

In [4]:
loader = next(iter(train_loader))

In [5]:
cell_type, cell_image, cell_segment_image, movie_id = loader

In [6]:
cell_image

tensor([[[[0.4392, 0.4510, 0.4588,  ..., 0.4471, 0.4588, 0.4824],
          [0.4784, 0.4510, 0.4549,  ..., 0.4667, 0.4863, 0.4549],
          [0.4667, 0.4706, 0.4431,  ..., 0.4706, 0.4549, 0.4392],
          ...,
          [0.4706, 0.4745, 0.4627,  ..., 0.4471, 0.4745, 0.4745],
          [0.4549, 0.4510, 0.4588,  ..., 0.4902, 0.4667, 0.4706],
          [0.4588, 0.4510, 0.4471,  ..., 0.4314, 0.4784, 0.4667]]],


        [[[0.4549, 0.4471, 0.4392,  ..., 0.4314, 0.4392, 0.4235],
          [0.4549, 0.4627, 0.4510,  ..., 0.4392, 0.4471, 0.4824],
          [0.4471, 0.4627, 0.4471,  ..., 0.4471, 0.4588, 0.5137],
          ...,
          [0.4627, 0.4627, 0.4627,  ..., 0.4627, 0.4588, 0.4549],
          [0.4588, 0.4824, 0.4667,  ..., 0.4745, 0.4745, 0.4824],
          [0.4667, 0.4745, 0.4706,  ..., 0.4667, 0.4745, 0.4706]]],


        [[[0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
          [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
          [0.0000, 0.0000, 0.0000,  ..